# 🔍 NER + LLM Expansion Pipeline
### Real-Time Query Expansion & Topic Tagging — Stage 3 (Inference)

**This notebook implements the full real-time inference system:**
```
Incoming user message
      │
      ├─► Sliding window (last 20 messages)
      │
      ├─► spaCy NER  →  Entity Register
      │                 (PERSON, GPE, ORG, EVENT)
      │
      ├─► Interruption detector  →  tag GENERAL, skip expansion
      │
      ├─► LLM rewriting prompt (Llama-3.2-1B-Instruct)
      │       context window + entity register → expanded query
      │
      └─► DistilBERT classifier  →  [TOPIC: L1 > L2]
```
**NER:** `en_core_web_trf` (spaCy transformer model — best accuracy)  
**LLM:** `meta-llama/Llama-3.2-1B-Instruct` (HuggingFace)  
**Classifier:** Your fine-tuned DistilBERT from the previous notebook


## 0. Install

In [1]:
!pip install transformers accelerate spacy torch huggingface_hub -q
# spaCy transformer NER model (best accuracy)
!python -m spacy download en_core_web_trf -q
# Fallback: lighter model if trf is too slow on CPU
# !python -m spacy download en_core_web_sm -q
print('✅ done')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 51.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ done


## 1. Config & HF Login

In [ ]:
import os, json, re, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from collections import deque
from dataclasses import dataclass, field
from typing import List, Optional, Dict
import torch
from huggingface_hub import login

# ── CONFIG ───────────────────────────────────────────────────────
HF_TOKEN          = "HF_TOKEN"   # <── replace
LLM_MODEL_ID      = "meta-llama/Llama-3.2-1B-Instruct"

# Your fine-tuned classifiers (local path or HF repo id)
# L1_MODEL_PATH     = "models/topic_l1_classifier/best_model"
# L2_MODEL_PATH     = "models/topic_l2_classifier/best_model"
# Or load from HF Hub:
L1_MODEL_PATH  = "Adignite/query-topic-l1-classifier"
L2_MODEL_PATH  = "Adignite/query-topic-l2-classifier"

SPACY_MODEL       = "en_core_web_trf"   # swap to en_core_web_sm on CPU
CONTEXT_WINDOW    = 20                  # last N messages to consider
MAX_NEW_TOKENS    = 500
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"

# Small talk / interruption patterns
INTERRUPT_PATTERNS = [
    r'^\s*(brb|brt|back|ok|okay|k|thanks|thank you|got it|noted|alright|sure|'
    r'wait|hold on|one sec|give me a (min|sec|moment)|be right back|'
    r'i.?m back|coming back|just a min|afk)[\.!\?]?\s*$'
]

login(token=HF_TOKEN)
print(f"✅ Config ready | Device: {DEVICE}")
print(f"📦 LLM          : {LLM_MODEL_ID}")
print(f"📊 Classifier L1: {L1_MODEL_PATH}")
print(f"📊 Classifier L2: {L2_MODEL_PATH}")

✅ Config ready | Device: cuda
📦 LLM          : meta-llama/Llama-3.2-1B-Instruct
📊 Classifier L1: Adignite/query-topic-l1-classifier
📊 Classifier L2: Adignite/query-topic-l2-classifier


## 2. Load All Models

In [2]:
import spacy

print(f"Loading spaCy model: {SPACY_MODEL} ...")
nlp = spacy.load(SPACY_MODEL)
print(f"✅ spaCy loaded | Pipeline: {nlp.pipe_names}")

Loading spaCy model: en_core_web_trf ...
✅ spaCy loaded | Pipeline: ['transformer', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"Loading LLM: {LLM_MODEL_ID} ...")
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID, token=HF_TOKEN)
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
)
if DEVICE == "cpu":
    llm_model = llm_model.to(DEVICE)
llm_model.eval()

HAS_CHAT_TEMPLATE = hasattr(llm_tokenizer, 'apply_chat_template') and llm_tokenizer.chat_template is not None
print(f"✅ LLM loaded | Chat template: {HAS_CHAT_TEMPLATE}")

Loading LLM: meta-llama/Llama-3.2-1B-Instruct ...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

✅ LLM loaded | Chat template: True


In [7]:
from transformers import pipeline as hf_pipeline
import pickle

print("Loading DistilBERT classifiers ...")
clf_l1 = hf_pipeline('text-classification', model=L1_MODEL_PATH,
                      device=0 if DEVICE=="cuda" else -1)
clf_l2 = hf_pipeline('text-classification', model=L2_MODEL_PATH,
                      device=0 if DEVICE=="cuda" else -1)
print("✅ Both classifiers loaded.")

Loading DistilBERT classifiers ...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Both classifiers loaded.


## 3. Core Components

In [8]:
# ── ENTITY REGISTER ──────────────────────────────────────────────
# Maintains a rolling list of named entities seen in the context window.
# Entities are ranked by recency (most recent = higher priority).

ENTITY_TYPES = {"PERSON", "GPE", "ORG", "EVENT", "NORP", "FAC", "LOC"}

@dataclass
class EntityEntry:
    text:      str
    label:     str
    turn_idx:  int   # which turn it was found in

class EntityRegister:
    """Sliding-window entity register for a conversation."""

    def __init__(self):
        self.entries: List[EntityEntry] = []

    def update(self, text: str, turn_idx: int):
        """Run NER on text and add new entities to register."""
        doc = nlp(text)
        for ent in doc.ents:
            if ent.label_ in ENTITY_TYPES:
                # Avoid duplicates within same turn
                if not any(e.text.lower() == ent.text.lower() and e.turn_idx == turn_idx
                           for e in self.entries):
                    self.entries.append(EntityEntry(
                        text=ent.text, label=ent.label_, turn_idx=turn_idx
                    ))

    def prune(self, min_turn: int):
        """Remove entities from turns older than min_turn."""
        self.entries = [e for e in self.entries if e.turn_idx >= min_turn]

    def get_recent(self, n: int = 5) -> List[EntityEntry]:
        """Return n most recently mentioned entities."""
        return sorted(self.entries, key=lambda e: e.turn_idx, reverse=True)[:n]

    def as_context_string(self) -> str:
        """Format for LLM prompt injection."""
        recent = self.get_recent(8)
        if not recent:
            return "(none)"
        return ", ".join(f"{e.text} [{e.label}]" for e in recent)

    def get_most_recent_person(self) -> Optional[str]:
        """Returns the most recently mentioned PERSON entity."""
        for e in sorted(self.entries, key=lambda e: e.turn_idx, reverse=True):
            if e.label == "PERSON":
                return e.text
        return None

print("✅ EntityRegister ready.")

✅ EntityRegister ready.


In [10]:
# ── INTERRUPTION DETECTOR ────────────────────────────────────────

def is_interruption(text: str) -> bool:
    """
    Returns True if the message is small talk, filler, or interruption.
    These get tagged GENERAL and are returned as-is without LLM expansion.
    """
    t = text.strip().lower()
    # Very short (<=3 words) with no question word
    words = t.split()
    question_words = {'who','what','where','when','why','how','which','whose','whom'}
    if len(words) <= 3 and not any(w in question_words for w in words):
        return True
    # Pattern matching
    for pattern in INTERRUPT_PATTERNS:
        if re.match(pattern, t, re.IGNORECASE):
            return True
    return False

# Quick tests
tests = [
    ("brb",                     True),
    ("wait a sec",              True),
    ("back, i'm here",          True),
    ("thanks",                  True),
    ("who is the pm of india?", False),
    ("what about uk?",          False)
]
print("Interruption detector tests:")
all_pass = True
for text, expected in tests:
    result = is_interruption(text)
    status = '✅' if result == expected else '❌'
    if result != expected: all_pass = False
    print(f"  {status} '{text}' → {result}")
print(f"\n{'All tests passed ✅' if all_pass else 'Some tests failed ❌'}")

Interruption detector tests:
  ✅ 'brb' → True
  ✅ 'wait a sec' → True
  ✅ 'back, i'm here' → True
  ✅ 'thanks' → True
  ✅ 'who is the pm of india?' → False
  ✅ 'what about uk?' → False

All tests passed ✅


In [20]:
# ── LLM EXPANSION PROMPT ─────────────────────────────────────────

EXPANSION_SYSTEM = """Rewrite the latest user message into a fully self-contained query using context.
- Replace all pronouns and implicit references with explicit names.
- Expand incomplete questions, topic shifts, or comparisons into full sentences.
- Output ONLY the rewritten query. No quotes, no explanations. Return as-is if already self-contained."""


def build_expansion_user_prompt(history: list, current_msg: str, entity_register: EntityRegister) -> str:
    # Format last N turns of conversation history
    history_str = "\n".join(
        f"{'User' if t['role']=='user' else 'Assistant'}: {t['text']}"
        for t in history
    )
    entities_str = entity_register.as_context_string()

    return (
        f"Conversation context (last {len(history)} messages):\n"
        f"{history_str}\n\n"
        f"Named entities in context: {entities_str}\n\n"
        f"Latest user message: {current_msg}\n\n"
        f"Rewrite as a self-contained question:"
    )


def llm_expand(history: list, current_msg: str, entity_register: EntityRegister) -> str:
    """
    Calls Llama-3.2-1B-Instruct to expand the current user message
    using conversation history and entity register.
    """
    user_prompt = build_expansion_user_prompt(history, current_msg, entity_register)

    if HAS_CHAT_TEMPLATE:
        messages = [
            {"role": "system", "content": EXPANSION_SYSTEM},
            {"role": "user",   "content": user_prompt},
        ]
        # Force return_dict=True to standardize output, then extract the tensor
        encoded = llm_tokenizer.apply_chat_template(
            messages, return_tensors="pt", add_generation_prompt=True, return_dict=True
        ).to(DEVICE)
        input_ids = encoded["input_ids"]
    else:
        combined  = f"### System:\n{EXPANSION_SYSTEM}\n\n### User:\n{user_prompt}\n\n### Response:\n"
        encoded = llm_tokenizer(combined, return_tensors="pt").to(DEVICE)
        input_ids = encoded["input_ids"]

    prompt_len = input_ids.shape[-1]

    with torch.no_grad():
        output_ids = llm_model.generate(
            input_ids,
            attention_mask    = encoded["attention_mask"], # Add this line
            max_new_tokens    = MAX_NEW_TOKENS,
            temperature       = 0.3,
            do_sample         = True,
            pad_token_id      = llm_tokenizer.pad_token_id,
            eos_token_id      = llm_tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][prompt_len:]
    expanded   = llm_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Clean up: take only the first line (model sometimes adds explanation)
    expanded = expanded.split('\n')[0].strip()
    # Remove common prefixes the model might add
    for prefix in ["Rewritten:", "Answer:", "Query:", "Here is", "Here's", "The question is"]:
        if expanded.lower().startswith(prefix.lower()):
            expanded = expanded[len(prefix):].strip()

    return expanded if expanded else current_msg  # fallback to original

print("✅ LLM expansion function ready.")

✅ LLM expansion function ready.


## 4. Conversation State Manager

In [21]:
# ── CONVERSATION STATE ────────────────────────────────────────────

@dataclass
class TurnResult:
    raw_message:     str
    expanded_query:  str
    topic_l1:        str
    topic_l1_score:  float
    topic_l2:        str
    topic_l2_score:  float
    entities_used:   List[str]
    was_expanded:    bool
    is_interruption: bool

    def display(self):
        print(f"  Raw      : {self.raw_message}")
        print(f"  Expanded : {self.expanded_query}")
        print(f"  Topic    : {self.topic_l1} > {self.topic_l2}  "
              f"(L1: {self.topic_l1_score:.2f}, L2: {self.topic_l2_score:.2f})")
        print(f"  Entities : {self.entities_used}")
        print(f"  Expanded?: {self.was_expanded} | Interrupt: {self.is_interruption}")


class ConversationProcessor:
    """
    Real-time conversation processor.
    Maintains a sliding window of the last CONTEXT_WINDOW messages,
    a rolling entity register, and processes each user turn end-to-end.
    """

    def __init__(self):
        self.history:   deque = deque(maxlen=CONTEXT_WINDOW)  # last 20 messages
        self.entities:  EntityRegister = EntityRegister()
        self.turn_idx:  int = 0

    def add_assistant_turn(self, text: str):
        """Add assistant response to history + NER it."""
        self.history.append({"role": "assistant", "text": text})
        self.entities.update(text, self.turn_idx)
        self.turn_idx += 1

    def process_user_turn(self, text: str) -> TurnResult:
        """
        Full pipeline for one user message:
        1. NER update
        2. Interruption check
        3. LLM expansion (if not interruption)
        4. BERT classification
        5. Return TurnResult
        """
        # Update entity register with user message
        self.entities.update(text, self.turn_idx)
        # Prune entities older than context window
        min_turn = max(0, self.turn_idx - CONTEXT_WINDOW)
        self.entities.prune(min_turn)

        entities_used = [e.text for e in self.entities.get_recent(5)]
        interrupt     = is_interruption(text)

        if interrupt:
            # Small talk: no expansion, tag as General
            expanded = text
            l1, l1s  = "General", 1.0
            l2, l2s  = "General", 1.0
            was_exp  = False
        else:
            # LLM expansion using history + entities
            history_list = list(self.history)  # snapshot before adding this turn
            expanded = llm_expand(history_list, text, self.entities)
            was_exp  = expanded.lower().strip() != text.lower().strip()

            # BERT classification on expanded query
            r1   = clf_l1(expanded)[0]
            r2   = clf_l2(expanded)[0]
            l1   = r1['label']
            l1s  = r1['score']
            l2   = r2['label']
            l2s  = r2['score']

        # Add user turn to history
        self.history.append({"role": "user", "text": text})
        self.turn_idx += 1

        return TurnResult(
            raw_message     = text,
            expanded_query  = expanded,
            topic_l1        = l1,
            topic_l1_score  = l1s,
            topic_l2        = l2,
            topic_l2_score  = l2s,
            entities_used   = entities_used,
            was_expanded    = was_exp,
            is_interruption = interrupt,
        )

    def reset(self):
        self.history.clear()
        self.entities = EntityRegister()
        self.turn_idx = 0

print("✅ ConversationProcessor ready.")

✅ ConversationProcessor ready.


## 5. Unit Tests — NER & Interruption Detector

In [22]:
print("── NER Tests ────────────────────────────────────")
ner_tests = [
    "Narendra Modi is the Prime Minister of India.",
    "Virat Kohli scored a century in the test match.",
    "OpenAI released GPT-4 in March 2023.",
    "Shah Rukh Khan starred in Jawan directed by Atlee.",
]
for text in ner_tests:
    doc = nlp(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents if ent.label_ in ENTITY_TYPES]
    print(f"  Text     : {text[:60]}")
    print(f"  Entities : {entities}\n")

── NER Tests ────────────────────────────────────
  Text     : Narendra Modi is the Prime Minister of India.
  Entities : [('Narendra Modi', 'PERSON'), ('India', 'GPE')]

  Text     : Virat Kohli scored a century in the test match.
  Entities : [('Virat Kohli', 'PERSON')]

  Text     : OpenAI released GPT-4 in March 2023.
  Entities : [('OpenAI', 'ORG')]

  Text     : Shah Rukh Khan starred in Jawan directed by Atlee.
  Entities : [('Shah Rukh Khan', 'PERSON'), ('Atlee', 'PERSON')]



In [23]:
print("── Expansion Tests (standalone) ─────────────────")

# Build a mini entity register for testing
test_register = EntityRegister()
test_register.update("Narendra Modi is the PM of India.", turn_idx=0)
test_register.update("He leads the Union Cabinet.", turn_idx=1)

test_history = [
    {"role": "user",      "text": "who is the pm of india?"},
    {"role": "assistant", "text": "Narendra Modi has been PM since 2014."},
    {"role": "user",      "text": "what are his responsibilities?"},
    {"role": "assistant", "text": "He leads the Cabinet and represents India globally."},
]

expansion_tests = [
    "what are his duties?",
    "what about uk?",
    "compare both",
    "hmm and internationally?",
    "who is the president of usa?",  # already self-contained
]

for query in expansion_tests:
    expanded = llm_expand(test_history, query, test_register)
    changed  = expanded.lower().strip() != query.lower().strip()
    print(f"  Raw      : {query}")
    print(f"  Expanded : {expanded}")
    print(f"  Changed  : {changed}\n")

── Expansion Tests (standalone) ─────────────────
  Raw      : what are his duties?
  Expanded : Who is responsible for the duties of the Prime Minister of India?
  Changed  : True

  Raw      : what about uk?
  Expanded : Who is the Prime Minister of the United Kingdom?
  Changed  : True

  Raw      : compare both
  Expanded : Who is compared to Narendra Modi in terms of his responsibilities as the PM of India?
  Changed  : True

  Raw      : hmm and internationally?
  Expanded : Who is the Prime Minister of India and what are his responsibilities?
  Changed  : True

  Raw      : who is the president of usa?
  Expanded : Who is the President of the United States of America?
  Changed  : True



## 6. Full Conversation Simulation (End-to-End Test)

In [24]:
# ── CONVERSATION 1: Politics + Interruption + Topic Switch ───────
print("\n" + "="*62)
print("  CONVERSATION 1: Politics → Interruption → UK switch")
print("="*62)

proc = ConversationProcessor()

dialogue = [
    # (role, text)
    ("user",      "who is pm of india?"),
    ("assistant", "The Prime Minister of India is Narendra Modi, in office since 2014."),
    ("user",      "what are his duties?"),
    ("assistant", "He leads the Cabinet, sets policy, and represents India internationally."),
    ("user",      "and internationally?"),
    ("assistant", "He attends G20, UN summits and handles bilateral diplomacy."),
    ("user",      "wait a sec"),                    # interruption
    ("assistant", "Sure, take your time!"),
    ("user",      "back, what about uk?"),           # topic switch
    ("assistant", "The Prime Minister of the UK is Rishi Sunak."),
    ("user",      "compare both"),                  # multi-entity
    ("assistant", "Modi leads India's BJP; Sunak leads UK's Conservatives."),
]

for role, text in dialogue:
    if role == "assistant":
        proc.add_assistant_turn(text)
        print(f"  Assistant : {text}")
    else:
        result = proc.process_user_turn(text)
        print(f"\n  ── Turn ──")
        result.display()

print()


  CONVERSATION 1: Politics → Interruption → UK switch

  ── Turn ──
  Raw      : who is pm of india?
  Expanded : "Who is the Prime Minister of India?"
  Topic    : Politics > India  (L1: 0.84, L2: 0.85)
  Entities : ['pm of india']
  Expanded?: True | Interrupt: False
  Assistant : The Prime Minister of India is Narendra Modi, in office since 2014.

  ── Turn ──
  Raw      : what are his duties?
  Expanded : What are the duties of the Prime Minister of India?
  Topic    : Politics > India  (L1: 0.83, L2: 0.85)
  Entities : ['India', 'Narendra Modi', 'pm of india']
  Expanded?: True | Interrupt: False
  Assistant : He leads the Cabinet, sets policy, and represents India internationally.

  ── Turn ──
  Raw      : and internationally?
  Expanded : and internationally?
  Topic    : General > General  (L1: 1.00, L2: 1.00)
  Entities : ['Cabinet', 'India', 'India', 'Narendra Modi', 'pm of india']
  Expanded?: False | Interrupt: True
  Assistant : He attends G20, UN summits and handles bil

In [25]:
# ── CONVERSATION 2: Sports ───────────────────────────────────────
print("\n" + "="*62)
print("  CONVERSATION 2: Sports (Cricket → Football)")
print("="*62)

proc2 = ConversationProcessor()

dialogue2 = [
    ("user",      "who is the best cricket player?"),
    ("assistant", "Virat Kohli is widely regarded as one of the greatest batsmen today."),
    ("user",      "how many centuries does he have?"),
    ("assistant", "Kohli has over 80 international centuries across all formats."),
    ("user",      "what about ms dhoni?"),           # topic switch within cricket
    ("assistant", "MS Dhoni is famous for his captaincy; he led India to the 2011 World Cup."),
    ("user",      "compare both their batting styles"),
    ("assistant", "Kohli is aggressive; Dhoni is famous for his finishing calm."),
    ("user",      "brb"),                            # interruption
    ("assistant", "Take your time!"),
    ("user",      "ok back. now what about messi?"), # domain switch: football
    ("assistant", "Lionel Messi is an Argentine footballer and 8-time Ballon d'Or winner."),
    ("user",      "how many world cups has he won?"),
    ("assistant", "Messi won his first FIFA World Cup with Argentina in 2022."),
]

for role, text in dialogue2:
    if role == "assistant":
        proc2.add_assistant_turn(text)
        print(f"  Assistant : {text}")
    else:
        result = proc2.process_user_turn(text)
        print(f"\n  ── Turn ──")
        result.display()


  CONVERSATION 2: Sports (Cricket → Football)

  ── Turn ──
  Raw      : who is the best cricket player?
  Expanded : Who is the best cricket player in the world?
  Topic    : Sports > Cricket  (L1: 0.88, L2: 0.79)
  Entities : []
  Expanded?: True | Interrupt: False
  Assistant : Virat Kohli is widely regarded as one of the greatest batsmen today.

  ── Turn ──
  Raw      : how many centuries does he have?
  Expanded : Who is Virat Kohli and how many centuries does he have?
  Topic    : Sports > Cricket  (L1: 0.93, L2: 0.90)
  Entities : ['Virat Kohli']
  Expanded?: True | Interrupt: False
  Assistant : Kohli has over 80 international centuries across all formats.

  ── Turn ──
  Raw      : what about ms dhoni?
  Expanded : Who is MS Dhoni?
  Topic    : Sports > Cricket  (L1: 0.92, L2: 0.89)
  Entities : ['dhoni', 'Kohli', 'Virat Kohli']
  Expanded?: True | Interrupt: False
  Assistant : MS Dhoni is famous for his captaincy; he led India to the 2011 World Cup.

  ── Turn ──
  Raw    

## 7. Batch Evaluation on Test Set

In [28]:
import pandas as pd
from sklearn.metrics import classification_report

# Load flat test set from BERT notebook split
TEST_CSV = "turns_dataset.csv"
df_test  = pd.read_csv(TEST_CSV).dropna(subset=['expanded_query','topic_l1'])

# Use 20% as test
from sklearn.model_selection import train_test_split
_, df_eval = train_test_split(df_test, test_size=0.2, random_state=42, stratify=df_test['topic_l1'])
df_eval = df_eval.reset_index(drop=True)
print(f"Evaluating on {len(df_eval)} samples...")

# Run classifier on expanded_queries (ground truth expansions from dataset)
# This tests the classifier independently of the LLM expansion quality
batch_size = 64
all_preds_l1 = []

for i in range(0, len(df_eval), batch_size):
    batch = df_eval['expanded_query'].iloc[i:i+batch_size].tolist()
    preds = clf_l1(batch)
    all_preds_l1.extend([p['label'] for p in preds])

print("\n── L1 Classification Report (on expanded queries) ──")
print(classification_report(df_eval['topic_l1'].tolist(), all_preds_l1, zero_division=0))

Evaluating on 410 samples...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



── L1 Classification Report (on expanded queries) ──
               precision    recall  f1-score   support

Entertainment       1.00      0.96      0.98        50
      General       0.68      0.94      0.79        18
    Geography       1.00      0.84      0.91        19
       Health       0.98      1.00      0.99        42
      History       1.00      0.90      0.95        21
     Politics       0.97      0.93      0.95        92
       Sports       0.98      0.98      0.98        90
   Technology       0.96      0.99      0.97        78

     accuracy                           0.96       410
    macro avg       0.95      0.94      0.94       410
 weighted avg       0.96      0.96      0.96       410



## 8. NER Quality Analysis

In [29]:
import ast
from sklearn.metrics import precision_score, recall_score, f1_score

# Compare spaCy NER output vs ground truth named_entities from dataset
df_ner = pd.read_csv(TEST_CSV).dropna(subset=['expanded_query','named_entities'])
df_ner = df_ner.head(200)  # sample 200 rows for speed

def extract_entities_spacy(text):
    doc = nlp(text)
    return set(ent.text.lower() for ent in doc.ents if ent.label_ in ENTITY_TYPES)

def parse_gt_entities(s):
    try:
        lst = ast.literal_eval(s) if isinstance(s, str) else s
        return set(e.lower() for e in lst) if lst else set()
    except:
        return set()

precisions, recalls, f1s = [], [], []
for _, row in df_ner.iterrows():
    pred = extract_entities_spacy(row['expanded_query'])
    gt   = parse_gt_entities(row['named_entities'])
    if not gt:
        continue
    tp = len(pred & gt)
    fp = len(pred - gt)
    fn = len(gt - pred)
    p  = tp / (tp + fp) if (tp + fp) > 0 else 0
    r  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f  = 2*p*r/(p+r) if (p+r) > 0 else 0
    precisions.append(p)
    recalls.append(r)
    f1s.append(f)

print("── NER Quality vs Ground Truth ──")
print(f"  Precision (avg) : {sum(precisions)/len(precisions)*100:.1f}%")
print(f"  Recall    (avg) : {sum(recalls)/len(recalls)*100:.1f}%")
print(f"  F1        (avg) : {sum(f1s)/len(f1s)*100:.1f}%")
print(f"  Samples   eval  : {len(f1s)}")

── NER Quality vs Ground Truth ──
  Precision (avg) : 59.6%
  Recall    (avg) : 58.4%
  F1        (avg) : 58.5%
  Samples   eval  : 129


## 9. Interactive Demo

In [31]:
# ── INTERACTIVE CONVERSATION ──────────────────────────────────────
# Type your messages below. Type 'quit' to exit, 'reset' to start a new conversation.
# For bot responses, type: bot: <response text>

proc_live = ConversationProcessor()

print("Interactive mode. Commands:")
print("  <your message>      → process as user turn")
print("  bot: <response>     → add assistant response to context")
print("  reset               → clear conversation")
print("  quit                → exit")
print("-" * 55)

while True:
    try:
        user_input = input("\n> ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nExiting.")
        break

    if not user_input:
        continue
    if user_input.lower() == 'quit':
        print("Goodbye!")
        break
    if user_input.lower() == 'reset':
        proc_live.reset()
        print("✅ Conversation reset.")
        continue
    if user_input.lower().startswith("bot:"):
        bot_text = user_input[4:].strip()
        proc_live.add_assistant_turn(bot_text)
        print(f"  [Context updated with bot response]")
        continue

    # Process user turn
    result = proc_live.process_user_turn(user_input)
    print()
    result.display()

Interactive mode. Commands:
  <your message>      → process as user turn
  bot: <response>     → add assistant response to context
  reset               → clear conversation
  quit                → exit
-------------------------------------------------------

> who is the best football player?

  Raw      : who is the best football player?
  Expanded : Who is the best football player in the world?
  Topic    : Sports > Football  (L1: 0.85, L2: 0.77)
  Entities : []
  Expanded?: True | Interrupt: False

> who is the best Prime minister?

  Raw      : who is the best Prime minister?
  Expanded : Who is the current Prime Minister of the United Kingdom?
  Topic    : Politics > UK  (L1: 0.88, L2: 0.80)
  Entities : []
  Expanded?: True | Interrupt: False

> who is the pm 12th pm of india

  Raw      : who is the pm 12th pm of india
  Expanded : Who is the 12th Prime Minister of India?
  Topic    : Politics > India  (L1: 0.87, L2: 0.86)
  Entities : []
  Expanded?: True | Interrupt: False

>